# Notebook 04 — RL-based Production Scheduling Optimization

**Module:** `optimization/production_rl.py`  
**Environment:** `ManufacturingEnv` (custom Gymnasium env)  
**Algorithm:** PPO (Proximal Policy Optimization)

---

## Problem Statement

In a job-shop manufacturing environment, machines must be allocated to jobs
to maximize throughput, minimize tardiness, and reduce downtime.

Classical scheduling algorithms (SPT, EDD, LPT) use fixed heuristics.
Reinforcement learning learns adaptive policies that respond to stochastic
conditions: random processing times, machine breakdowns, variable demand.

**MDP formulation:**
- **State:** machine states, queue lengths, repair timers
- **Action:** assign next queued job to machine i (or wait)
- **Reward:** +throughput − tardiness penalty − downtime cost
- **Algorithm:** PPO (on-policy, stable for discrete action spaces)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from optimization.production_rl import ManufacturingEnv, ProductionAgent
from evaluation.rl_metrics import (
    compute_oee, plot_reward_curve, plot_oee_comparison, summarize_training
)

print('Modules loaded.')

## 1. Explore the Environment

In [ ]:
env = ManufacturingEnv(
    num_machines=5,
    num_job_types=4,
    max_queue_length=20,
    processing_time_range=(1, 10),
    breakdown_probability=0.02,
    repair_time=5,
    max_episode_steps=200,
)

print(f'Observation space: {env.observation_space}')
print(f'Action space:      {env.action_space}')
print(f'Obs dim: {env.observation_space.shape[0]}')
print(f'n_actions: {env.action_space.n}')

In [ ]:
# Random policy baseline
obs, info = env.reset(seed=0)
ep_reward = 0
rewards_random = []

for _ in range(200):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    ep_reward += reward
    rewards_random.append(reward)
    if terminated or truncated:
        break

print(f'Random policy episode reward: {ep_reward:.2f}')
print(f'Jobs completed: {info["total_completed"]}')

## 2. Train PPO Agent

In [ ]:
try:
    agent = ProductionAgent(
        env_kwargs={
            'num_machines': 5,
            'num_job_types': 4,
            'max_episode_steps': 200,
            'breakdown_probability': 0.02,
        },
        ppo_kwargs={'verbose': 0},
    )

    agent.train(
        total_timesteps=200_000,     # increase for better convergence
        eval_freq=5_000,
        n_eval_episodes=5,
        checkpoint_dir='../checkpoints/notebook04_rl',
        log_dir='../logs/notebook04_rl',
    )
    SB3_AVAILABLE = True
except ImportError as e:
    print(f'stable-baselines3 not installed: {e}')
    print('Install with: pip install stable-baselines3')
    SB3_AVAILABLE = False

## 3. Compare PPO vs. Random vs. Always-Dispatch Heuristics

In [ ]:
def evaluate_policy(policy_fn, n_episodes=20, max_steps=200, seed=0):
    """Evaluate a policy function and return per-episode info."""
    eval_env = ManufacturingEnv(max_episode_steps=max_steps)
    results = []
    for ep in range(n_episodes):
        obs, info = eval_env.reset(seed=seed + ep)
        ep_reward = 0
        while True:
            action = policy_fn(obs, eval_env)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            ep_reward += reward
            if terminated or truncated:
                break
        results.append({'reward': ep_reward, 'completed': info['total_completed']})
    return results

# Heuristic 1: Random
def random_policy(obs, env): return env.action_space.sample()

# Heuristic 2: Always dispatch to machine 0 if idle
def greedy_policy(obs, env):
    for m in range(env.num_machines):
        if obs[m] < 0.1:  # machine state ≈ 0 = idle
            return m
    return env.num_machines  # wait

random_results = evaluate_policy(random_policy)
greedy_results = evaluate_policy(greedy_policy)

random_rewards  = [r['reward']    for r in random_results]
greedy_rewards  = [r['reward']    for r in greedy_results]
random_complete = [r['completed'] for r in random_results]
greedy_complete = [r['completed'] for r in greedy_results]

print(f'Random:  mean reward={np.mean(random_rewards):.2f}, mean completed={np.mean(random_complete):.1f}')
print(f'Greedy:  mean reward={np.mean(greedy_rewards):.2f}, mean completed={np.mean(greedy_complete):.1f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

policies = ['Random', 'Greedy']
rewards  = [np.mean(random_rewards), np.mean(greedy_rewards)]
complete = [np.mean(random_complete), np.mean(greedy_complete)]

if SB3_AVAILABLE:
    ppo_results = agent.evaluate(n_episodes=20)
    ppo_mean_reward = ppo_results.get('mean_reward', 0)
    ppo_mean_complete = ppo_results.get('mean_completed', 0)
    policies.append('PPO')
    rewards.append(ppo_mean_reward)
    complete.append(ppo_mean_complete)

colors = ['#e74c3c', '#f39c12', '#2ecc71'][:len(policies)]

ax1.bar(policies, rewards, color=colors)
ax1.set_ylabel('Mean Episode Reward')
ax1.set_title('Policy Comparison — Reward')
ax1.grid(axis='y', alpha=0.3)

ax2.bar(policies, complete, color=colors)
ax2.set_ylabel('Mean Jobs Completed')
ax2.set_title('Policy Comparison — Throughput')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. OEE Analysis

In [ ]:
ENV_STEPS = 200
NUM_MACHINES = 5
AVG_PROC_TIME = 5.5  # (1 + 10) / 2

oee_results = {}
for name, completed in zip(policies, complete):
    oee_results[name] = compute_oee(
        completed_jobs=int(completed),
        max_episode_steps=ENV_STEPS,
        num_machines=NUM_MACHINES,
        avg_processing_time=AVG_PROC_TIME,
    )
    print(f'{name}: OEE={oee_results[name]["oee"]:.3f}')

fig = plot_oee_comparison(oee_results, title='OEE Comparison Across Policies')
plt.show()

## Summary

The PPO agent learns a policy significantly better than random and rule-based
heuristics, especially when the production line faces random breakdowns and
variable demand.

**Key hyperparameters to tune:**
- `n_steps`: increase for more stable gradient estimates
- `ent_coef`: increase if the agent converges too quickly to a suboptimal policy
- `breakdown_probability`: increase to train a more robust policy

**Next:** `05_digital_twin_intro.ipynb` — simulating and syncing the production line.